# Finding Relationship between Discount and Rating of a Tokopedia product

In [55]:
import pandas as pd
import ast
import numpy as np
from scipy.stats import chi2_contingency
import plotly.express as px
import statsmodels.formula.api as smf
df_raw = pd.read_csv("tokopedia_products_with_review.csv")

## Filtering columns and omitting the NA part

In [56]:
columns = ["product_id", "category", "price", "discounted_price",
           "rating_average", "review_rating", "count_sold"]
df = df_raw[columns].copy()
df.info()
print(df[["rating_average", "count_sold"]].isna().value_counts())
df_main = df.dropna(subset=["rating_average"])
length_df_main = len(df_main)
print(length_df_main)
print(df["product_id"].nunique(), len(df))
dup = df[df["product_id"].duplicated(keep=False)].sort_values("product_id")
print(len(dup))
dup.head(10)
unique_df = df.drop_duplicates(subset=["product_id"]).copy()
unique_df = unique_df.dropna(subset=["rating_average"]).copy()
unique_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5553 entries, 0 to 5552
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        5553 non-null   int64  
 1   category          5553 non-null   str    
 2   price             5553 non-null   int64  
 3   discounted_price  1340 non-null   float64
 4   rating_average    5487 non-null   float64
 5   review_rating     5553 non-null   str    
 6   count_sold        4743 non-null   float64
dtypes: float64(3), int64(2), str(2)
memory usage: 4.2 MB
rating_average  count_sold
False           False         4731
                True           756
True            True            54
                False           12
Name: count, dtype: int64
5487
5410 5553
279
<class 'pandas.DataFrame'>
Index: 5344 entries, 0 to 5552
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        5344 

## Feature Engineering Ygy

In [57]:
unique_df["ratings"] = unique_df["review_rating"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
unique_df["n_reviews"] = unique_df["ratings"].apply(len)
unique_df["mean_rating"] = unique_df["ratings"].apply(lambda x: sum(x) / len(x) if len(x) > 0 else None)
unique_df["low_share"] = unique_df["ratings"].apply(lambda x: sum(r <= 2 for r in x) / len(x) if len(x) > 0 else None)
unique_df["has_discount"] = unique_df["discounted_price"].notna().astype(int)
unique_df["discount_percentage"] = (unique_df["discounted_price"] - unique_df["price"]) / unique_df["discounted_price"] * 100
unique_df["main_category"] = unique_df["category"].str.split("|").str[0].str.strip()
df_discount_only = unique_df[unique_df["has_discount"] == 1].copy()
unique_df["n_log_reviews"] = np.log10(unique_df["n_reviews"] + 1)
print(unique_df[["mean_rating", "discount_percentage", "low_share", "n_reviews"]].describe())
print(unique_df.loc[unique_df["has_discount"] == 1, "discount_percentage"].describe())
d = unique_df[unique_df["has_discount"] == 1]
print((d["discount_percentage"] < 1).sum())
print(d["discount_percentage"].round().value_counts().head(10))
print(pd.cut(d["discount_percentage"], bins=range(0, 101, 10)).value_counts().sort_index())
print(pd.cut(d["discount_percentage"], bins=range(0, 101, 10)).value_counts().sort_index().to_string())
x = d[d["discount_percentage"].between(60, 70)]
print(x["discount_percentage"].round().value_counts().head(10).to_string())


       mean_rating  discount_percentage    low_share    n_reviews
count  4070.000000          1266.000000  4070.000000  5344.000000
mean      4.896121            29.569565     0.013604   229.930015
std       0.222292            21.770215     0.045789   849.915775
min       1.000000             0.000008     0.000000     0.000000
25%       4.857143            10.000000     0.000000     1.000000
50%       5.000000            25.099621     0.000000     4.000000
75%       5.000000            47.888444     0.013035    60.250000
max       5.000000            87.693103     1.000000  9757.000000
count    1266.000000
mean       29.569565
std        21.770215
min         0.000008
25%        10.000000
50%        25.099621
75%        47.888444
max        87.693103
Name: discount_percentage, dtype: float64
44
discount_percentage
5.0     85
10.0    64
25.0    52
20.0    46
15.0    43
0.0     42
50.0    35
30.0    32
26.0    24
8.0     24
Name: count, dtype: int64
discount_percentage
(0, 10]      326


## Hypothesis Testing (Chi-Square)

In [58]:
for cut in [4.5, 4.8, 5.0]:
  rating_good_enough = unique_df["rating_average"] >= cut
  df_table_cont = pd.crosstab(unique_df["has_discount"], rating_good_enough)
  print(df_table_cont)
  chi2, p, dof, expected = chi2_contingency(df_table_cont) 
  print(f"Chi-square statistic: {chi2}, p-value: {p}")
##ambil cut = 5 sih, dimana disini menunjukkan bahwa memiliki asosisasi yang signifikan antara memiliki diskon dan memiliki rating 5.0, tolak H0
print(unique_df[["has_discount", "n_log_reviews"]].corr())
##corr masih aman bre

rating_average  False  True 
has_discount                
0                   9   4069
1                   3   1263
Chi-square statistic: 0.0, p-value: 1.0
rating_average  False  True 
has_discount                
0                  47   4031
1                  30   1236
Chi-square statistic: 9.23918700607658, p-value: 0.0023689010591128293
rating_average  False  True 
has_discount                
0                 550   3528
1                 696    570
Chi-square statistic: 927.7808507684158, p-value: 8.96806116280507e-204
               has_discount  n_log_reviews
has_discount       1.000000       0.480146
n_log_reviews      0.480146       1.000000


## Logistic Regression

In [59]:
unique_df["is_5"] = (unique_df["rating_average"] == 5.0).astype(int)
unique_df["disc_pct10"] = unique_df["discount_percentage"].fillna(0)
unique_df["pricelist"] = unique_df["discounted_price"].fillna(unique_df["price"])
unique_df["log_pricelist"] = np.log10(unique_df["pricelist"]) ##biar enggak se skew itu
unique_df["log_sold"] = np.log10(unique_df["count_sold"] + 1)  ## +1 untuk avoid log(0)
sub = unique_df.dropna(subset=["count_sold"]).copy()
big = unique_df[unique_df["n_reviews"] >= 10]
m1 = smf.logit("is_5 ~ has_discount", data=unique_df).fit()
m2 = smf.logit("is_5 ~ has_discount + n_log_reviews", data=unique_df).fit()
m3 = smf.logit("is_5  ~ has_discount + n_log_reviews + log_pricelist", data=unique_df).fit()
m4 = smf.logit("is_5 ~ has_discount + n_log_reviews + log_pricelist", data=sub).fit()
m5 = smf.logit("is_5 ~ has_discount + n_log_reviews + log_pricelist + log_sold", data=sub).fit()
m6 = smf.logit("is_5 ~ has_discount + n_log_reviews + log_pricelist", data=big).fit()
print(unique_df.assign(kosong=unique_df["count_sold"].isna())
      .groupby("kosong")[["has_discount", "is_5", "n_reviews"]].mean())
for m in (m1, m2, m3, m4, m5, m6):
    print(m.nobs)
    print(m.summary().tables[1])
    print(np.exp(m.params).round(3))


import plotly.express as px

grid = np.linspace(0, unique_df["n_log_reviews"].max(), 100)
price_mid = unique_df["log_pricelist"].median()

new = pd.concat([
    pd.DataFrame({"has_discount": g, "n_log_reviews": grid, "log_pricelist": price_mid})
    for g in (0, 1)
])
new["p"] = m3.predict(new)
new["grup"] = new["has_discount"].map({0: "Tanpa diskon", 1: "Diskon"})

fig = px.line(new, x="n_log_reviews", y="p", color="grup",
              labels={"n_log_reviews": "log10(jumlah review + 1)", "p": "Peluang rating 5.0"})
fig.update_yaxes(range=[0, 1])
fig.show()



Optimization terminated successfully.
         Current function value: 0.464869
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.337645
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.336406
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.318807
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.313588
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.605366
         Iterations 5
        has_discount      is_5   n_reviews
kosong                                    
False       0.173696  0.764130  247.550870
True        0.627688  0.783602  120.983871
5344
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        1.8586      0.046     40.542    

## Graph


In [60]:
rob = pd.DataFrame({
    "model": ["m1 mentah", "m2 + review", "m3 + harga", "m6 review ≥10"],
    "odds_ratio": [np.exp(m.params["has_discount"]) for m in (m1, m2, m3, m6)],
})
fig2 = px.bar(rob, x="model", y="odds_ratio", text_auto=".2f",
              labels={"odds_ratio": "Odds ratio diskon (1 = tidak ada efek)"})
fig2.add_hline(y=1, line_dash="dash")
fig2.show()

In [61]:
tmp = unique_df.copy()
tmp["bin_review"] = pd.cut(tmp["n_reviews"], [-1, 0, 2, 5, 10, 30, 100, 10_000],
                           labels=["0", "1-2", "3-5", "6-10", "11-30", "31-100", ">100"])
emp = tmp.groupby(["bin_review", "has_discount"], observed=True)["is_5"].agg(["mean", "size"]).reset_index()
emp["grup"] = emp["has_discount"].map({0: "Tanpa diskon", 1: "Diskon"})
fig3 = px.line(emp, x="bin_review", y="mean", color="grup", markers=True,
               labels={"mean": "Persen berating 5.0", "bin_review": "Jumlah review"})
fig3.update_yaxes(range=[0, 1])
fig3.show()
print(emp[["bin_review", "grup", "size"]].to_string())   # cek: kelompok dengan n kecil = garisnya berisik

   bin_review          grup  size
0           0  Tanpa diskon  1201
1           0        Diskon    73
2         1-2  Tanpa diskon  1090
3         1-2        Diskon    95
4         3-5  Tanpa diskon   397
5         3-5        Diskon    61
6        6-10  Tanpa diskon   234
7        6-10        Diskon    60
8       11-30  Tanpa diskon   361
9       11-30        Diskon   140
10     31-100  Tanpa diskon   334
11     31-100        Diskon   192
12       >100  Tanpa diskon   461
13       >100        Diskon   645


In [62]:
raw = unique_df.groupby("has_discount")["is_5"].mean().reset_index()
raw["grup"] = raw["has_discount"].map({0: "Tanpa diskon", 1: "Diskon"})
px.bar(raw, x="grup", y="is_5", text_auto=".1%").show()